In [5]:
pip install feedparser requests beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0 -> 26.0
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


RSS einlesen

In [6]:
import feedparser

RSS_URL = "https://correctiv.org/feed/"

feed = feedparser.parse(RSS_URL)

print("entries:", len(feed.entries))


ConnectionResetError: [Errno 54] Connection reset by peer

Einträge extrahieren

In [3]:
def parse_rss(feed):
    items = []
    for e in feed.entries:
        items.append({
            "title": e.get("title"),
            "url": e.get("link"),
            "published": e.get("published"),
            "author": e.get("author"),
            "categories": [t["term"] for t in e.get("tags", [])],
            "summary": e.get("summary"),
        })
    return items

rss_items = parse_rss(feed)


In [4]:
print(rss_items)

[{'title': 'Bundesamt für Migration und Flüchtlinge stoppt Zulassungen für Integrationskurse', 'url': 'https://correctiv.org/aktuelles/integration-gesellschaft/2026/02/04/bundesamt-fuer-migration-und-fluechtlinge-stoppt-zulassungen-fuer-integrationskurse/', 'published': 'Wed, 04 Feb 2026 09:59:57 +0000', 'author': 'Anette Dowideit', 'categories': ['Flucht & Migration', 'Gesellschaft', 'Integration & Gesellschaft', 'BAMF', 'Bundesamt für Migration', 'Featured-auf-Startseite', 'Integration', 'Migration', 'Migrationspolitik'], 'summary': 'Seit Dezember hat das BAMF nach Informationen von CORRECTIV die Genehmigungen für Integrationskurse auf Eis gelegt. Träger berichten, dies habe auch Auswirkungen auf Menschen, deren Kurse bereits genehmigt wurden.'}, {'title': 'Die russische Aktivistin in Epsteins Haus', 'url': 'https://correctiv.org/aktuelles/russland-ukraine-2/2026/02/04/die-russische-aktivistin-in-epsteins-haus/', 'published': 'Wed, 04 Feb 2026 05:58:30 +0000', 'author': 'Anette Dowid

Volltext aus den Artikeln holen

In [5]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (ThesisScraper/1.0; academic research)"
}

def fetch_article(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.select_one("article")
    if not article:
        return None

    paragraphs = article.select("p")
    text = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return text


/Users/gretawette/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [6]:
results = []

for item in rss_items:
    try:
        text = fetch_article(item["url"])
        item["text"] = text
        results.append(item)
        time.sleep(1.2)  # wichtig!
    except Exception as e:
        item["error"] = str(e)
        results.append(item)


In [7]:
import json

with open("correctiv_articles.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


## Themenspezifisches Scraping

In [7]:
import time
import requests
import feedparser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RSS_URL = "https://correctiv.org/faktencheck/tag/klima/feed/"

def make_session():
    s = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    return s

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/121.0 Safari/537.36",
    "Accept": "application/rss+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}

session = make_session()

resp = session.get(RSS_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

feed = feedparser.parse(resp.text)

print("HTTP:", resp.status_code)
print("entries:", len(feed.entries))
print(feed.entries[0].get("title"), feed.entries[0].get("link"))


/Users/gretawette/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


HTTP: 200
entries: 100
Zustand des Great Barrier Reefs schwankt stark https://correctiv.org/faktencheck/2026/01/30/zustand-des-great-barrier-reefs-schwankt-stark/


In [8]:
def parse_rss(feed):
    items = []
    for e in feed.entries:
        items.append({
            "title": e.get("title"),
            "url": e.get("link"),
            "published": e.get("published"),
            "author": e.get("author"),
            "categories": [t["term"] for t in e.get("tags", [])],
            "summary": e.get("summary"),
        })
    return items

rss_items = parse_rss(feed)

In [9]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (ThesisScraper/1.0; academic research)"
}

def fetch_article(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.select_one("article")
    if not article:
        return None

    paragraphs = article.select("p")
    text = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return text


In [10]:
results = []

for item in rss_items:
    try:
        text = fetch_article(item["url"])
        item["text"] = text
        results.append(item)
        time.sleep(1.2)  # wichtig!
    except Exception as e:
        item["error"] = str(e)
        results.append(item)

In [11]:
import json

with open("correctiv_articles_klima.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")